# BIPN 162 Final Project
ByEmily McClung, Naomi Ortega, and Isabel Rodriguez

In [1]:
!pip install pynwb h5py torch scikit-learn configmypy tensorboardX

Defaulting to user installation because normal site-packages is not writeable


In [2]:
#CLone the DUNL algorithm GitHub Repository
!git clone https://github.com/btolooshams/dunl-compneuro.git

fatal: destination path 'dunl-compneuro' already exists and is not an empty directory.


In [5]:
import numpy as np
import torch
import sklearn
import h5py
import configmypy

#Import the DUNL code
import sys
sys.path.append("dunl-compneuro/dunl")
#import dunl

#Imports for data
import os
from pynwb import NWBHDF5IO
os.makedirs("data", exist_ok=True)

In [4]:
!ls dunl-compneuro/dunl

__pycache__	     preprocess_scripts
boardfunc.py	     train_fiber_loop_acrossneurons.py
datasetloader.py     train_independentkernels_acrossneurons.py
lossfunc.py	     train_sharekernels_acrossneurons.py
model.py	     train_sharekernels_acrossneurons_groupneuralfirings.py
postprocess_scripts  utils.py


## Applying DUNL to another dataset

We will now apply the DUNL analysis method to a dataset of Fiber Photometry and Optogenetics recordings.

In [18]:
#download the dataset DANDI 000971
!pip install dandi
#download the specific files with the data we need
#RI60 PR (habit formation mimic)
!/home/emcclung/.local/bin/dandi download "https://dandiarchive.org/dandiset/000971/0.260213.1851/files?location=sub-028-392" --output-dir data

!/home/emcclung/.local/bin/dandi download "https://dandiarchive.org/dandiset/000971/0.260213.1851/files?location=sub-140-306" --output-dir data

!/home/emcclung/.local/bin/dandi download "https://dandiarchive.org/dandiset/000971/0.260213.1851/files?location=sub-100-258" --output-dir data

#specfic files we will look at
keep_files = {
"sub-028-392_ses-FP-PR-2020-07-24T12-31-24.nwb",
"sub-028-392_ses-FP-PR-2020-07-09T13-01-26.nwb",
"sub-140-306_ses-FP-PS-2019-08-09T12-10-58_behavior.nwb",
"sub-140-306_ses-FP-PS-2019-09-03T10-15-44_behavior.nwb",
"sub-100-258_ses-FP-RR20-2019-05-09T13-32-40_behavior.nwb",
"sub-100-258_ses-FP-RR20-2019-04-23T12-25-00_behavior.nwb"
}

data_dir = "data"
#delete unused files 
for root, dirs, files in os.walk(data_dir):
    for f in files:
        if f.endswith(".nwb") and f not in keep_files:
            path = os.path.join(root, f)
            os.remove(path)
    print("Deleted unused files")

Defaulting to user installation because normal site-packages is not writeable
PATH                                                      SIZE      DONE            DONE% CHECKSUM STATUS          MESSAGE
sub-028-392/sub-028-392_ses-FP-PR-2020-06-19T12-45-31.nwb 232.0 kB  232.0 kB         100%    ok    done                   
sub-028-392/sub-028-392_ses-FP-PR-2020-06-16T13-05-50.nwb 224.4 kB  224.4 kB         100%    ok    done                   
sub-028-392/sub-028-392_ses-FP-PR-2020-06-17T13-09-45.nwb 229.9 kB  229.9 kB         100%    ok    done                   
sub-028-392/sub-028-392_ses-FP-PR-2020-06-26T13-41-50.nwb 229.9 kB  229.9 kB         100%    ok    done                   
sub-028-392/sub-028-392_ses-FP-PR-2020-06-27T13-51-35.nwb 229.9 kB  229.9 kB         100%    ok    done                   
sub-028-392/sub-028-392_ses-FP-PR-2020-06-28T13-14-50.nwb 229.9 kB  229.9 kB         100%    ok    done                   
sub-028-392/sub-028-392_ses-FP-PR-2020-07-01T13-57-44.nwb 232

In [6]:
#Group files by subject
028_dir = "data/sub-028-392"
files_028 = [f for f in os.listdir(028_dir) if f.endswith(".nwb")]

140_dir = "data/sub-140-306"
files_140 = [f for f in os.listdir(140_dir) if f.endswith(".nwb")]

100_dir = "data/sub-028-258"
files_100 = [f for f in os.listdir(100_dir) if f.endswith(".nwb")]

print(files_028)
print(files_140)
print(files_100)

['sub-028-392_ses-FP-PR-2020-07-24T12-31-24.nwb', 'sub-028-392_ses-FP-PR-2020-07-09T13-01-26.nwb']


In [7]:
#extracts dms_signal, dls_signal, and timestamps from files
def load_fp_signal(file_path):

    with NWBHDF5IO(file_path, "r") as io:
        nwb = io.read()

        fp_series = nwb.acquisition['fiber_photometry_response_series']
        raw_signal = fp_series.data[:]

        rate = fp_series.rate
        start = fp_series.starting_time
        timestamps = start + (np.arange(len(raw_signal)) / rate)

        dms_signal = raw_signal[:,0]
        dls_signal = raw_signal[:,2]

    return dms_signal, dls_signal, timestamps

# DUNL 1

In [8]:
#initalize lists
dms_sessions = []
dls_sessions = []

#loop through files and load signals
for f in files_028:
    file_path = os.path.join(dir_028, f)
    dms, dls, timestamps = load_fp_signal(file_path)
    dms_sessions.append(dms)
    dls_sessions.append(dls)

#convert lists to object arrays
dms_sessions = np.array(dms_sessions, dtype=object)
dls_sessions = np.array(dls_sessions, dtype=object)

#find the shortest session length to ensure uniform matrix dimensions
min_len = min(len(s) for s in dms_sessions)

#trim sessions to the minimum length and add the 'neuron' dimension (axis 1)
#shape: (num_trials, 1, trial_length)
dms_matrix = np.array([s[:min_len] for s in dms_sessions])[:, np.newaxis, :]
dls_matrix = np.array([s[:min_len] for s in dls_sessions])[:, np.newaxis, :]

#define dimensions for metadata
num_trials, num_neurons, trial_length = dls_matrix.shape

#fefine a helper to package and save as PyTorch tensors immediately
def save_as_tensor_dict(matrix, filename):
    tensor_dict = {
        "y": torch.as_tensor(matrix, dtype=torch.float32),
        "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
        "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
        "type": torch.zeros(num_trials, dtype=torch.int64)
    }
    torch.save(tensor_dict, filename)
    return tensor_dict

#save processed data
dms_sessions_fixed = save_as_tensor_dict(dms_matrix, "data/dms_sessions.pt")
dls_sessions_fixed = save_as_tensor_dict(dls_matrix, "data/dls_sessions.pt")

#verify types and shapes
for key, value in dls_sessions_fixed.items():
    print(f"DLS {key}: {type(value)} | Shape: {value.shape}")

/home/emcclung/.local/lib/python3.11/site-packages/hdmf/spec/namespace.py:484: UserWarning: Schema conflict(s) detected in namespace 'ndx-fiber-photometry': 
 ndx-fiber-photometry defines OpticalFiber.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines ExcitationSource.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines Photodetector.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines DichroicMirror.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines BandOpticalFilter.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines EdgeOpticalFilter.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel. 
T

Successfully converted all data to Tensors.
y: <class 'torch.Tensor'> | Shape: torch.Size([1, 1, 5000])
x: <class 'torch.Tensor'> | Shape: torch.Size([1, 1, 1])
a: <class 'torch.Tensor'> | Shape: torch.Size([1, 1, 1])
type: <class 'torch.Tensor'> | Shape: torch.Size([1])


In [10]:
#taking the first 15,000 time points so when we run we do not run out of memory
original_data = torch.as_tensor(dls_matrix, dtype=torch.float32)

if original_data.dim() == 2:
    y_small = original_data[:1, :15000].unsqueeze(0) 
else:
    y_small = original_data.flatten()[:15000].reshape(1, 1, 15000)

num_trials = y_small.shape[0]
num_neurons = y_small.shape[1]

dls_test_set = {
    "y": y_small,
    "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "type": torch.zeros(num_trials, dtype=torch.int64)
}

torch.save(dls_test_set, "data/dls_sessions.pt")

print(f"New Shape: {y_small.shape}")
print(f"Points per window: {y_small.shape[2] // 50}")

New Shape: torch.Size([1, 1, 15000])
Points per window: 300


Changes made to configurations in the dunl-compneuro repo:
1. In `config/dopamine_fiberphotometry_saramatias_uchida_config_1window_1kernel.yaml`
changed **from** `data_folder:  "../data/dopamine-spiking-eshel-uchida"`
**to** `data_folder: "data/dls_sessions.pt"`

2. In `dunl/train_fiber_loop_acrossneurons.py ` inside `init_params()`
 **set** `default="dunl-compneuro/config"`
to follow the correct path

3. In `dunl/datasetloader.py`
**added** `'import numpy'`
inside `init()`**added** `with torch.serialization.safe_globals([numpy.core.multiarray._reconstruct]):
            data = torch.load(data_path, weights_only=False)`



In [ ]:
!python dunl-compneuro/dunl/train_fiber_loop_acrossneurons.py 

init parameters.
neuron_index 0
Train DUNL on neural data (independent kernels for each neuron).
device is cpu
Exp: fiber_mouseX_dms
create dataset and dataloader.
There 1 dataset in the folder.
data/dls_sessions.pt
x is shared among neurons. It is a function of trials, and number of kernels!
create board.
x is shared among neurons. It is a function of trials, and number of kernels!
This script has a hardcoded step for dataset processing.
these are hardcoded now! should be moved into pre-processing step.
create model.
create optimizer and scheduler for training.
start training.
  3%|▉                                  | 26/1000 [1:42:40<65:39:18, 242.67s/it]

# DUNL 2

In [ ]:
#taking the first 15,000 time points so when we run we do not run out of memory
original_data = torch.as_tensor(dms_matrix, dtype=torch.float32)

if original_data.dim() == 2:
    y_small = original_data[:1, :15000].unsqueeze(0) 
else:
    y_small = original_data.flatten()[:15000].reshape(1, 1, 15000)

num_trials = y_small.shape[0]
num_neurons = y_small.shape[1]

dls_test_set = {
    "y": y_small,
    "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "type": torch.zeros(num_trials, dtype=torch.int64)
}

torch.save(dms_test_set, "data/dms_sessions.pt")

print(f"New Shape: {y_small.shape}")
print(f"Points per window: {y_small.shape[2] // 50}")

In [ ]:
!python dunl-compneuro/dunl/train_fiber_loop_acrossneurons.py 

# DUNL 3

In [ ]:
#initalize lists
dms_sessions = []
dls_sessions = []

#loop through files and load signals
for f in files_140:
    file_path = os.path.join(dir_140, f)
    dms, dls, timestamps = load_fp_signal(file_path)
    dms_sessions.append(dms)
    dls_sessions.append(dls)

#convert lists to object arrays
dms_sessions = np.array(dms_sessions, dtype=object)
dls_sessions = np.array(dls_sessions, dtype=object)

#find the shortest session length to ensure uniform matrix dimensions
min_len = min(len(s) for s in dms_sessions)

#trim sessions to the minimum length and add the 'neuron' dimension (axis 1)
#shape: (num_trials, 1, trial_length)
dms_matrix = np.array([s[:min_len] for s in dms_sessions])[:, np.newaxis, :]
dls_matrix = np.array([s[:min_len] for s in dls_sessions])[:, np.newaxis, :]

#define dimensions for metadata
num_trials, num_neurons, trial_length = dls_matrix.shape

#fefine a helper to package and save as PyTorch tensors immediately
def save_as_tensor_dict(matrix, filename):
    tensor_dict = {
        "y": torch.as_tensor(matrix, dtype=torch.float32),
        "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
        "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
        "type": torch.zeros(num_trials, dtype=torch.int64)
    }
    torch.save(tensor_dict, filename)
    return tensor_dict

#save processed data
dms_sessions_fixed = save_as_tensor_dict(dms_matrix, "data/dms_sessions.pt")
dls_sessions_fixed = save_as_tensor_dict(dls_matrix, "data/dls_sessions.pt")

#verify types and shapes
for key, value in dls_sessions_fixed.items():
    print(f"DLS {key}: {type(value)} | Shape: {value.shape}")

In [ ]:
#taking the first 15,000 time points so when we run we do not run out of memory
original_data = torch.as_tensor(dls_matrix, dtype=torch.float32)

if original_data.dim() == 2:
    y_small = original_data[:1, :15000].unsqueeze(0) 
else:
    y_small = original_data.flatten()[:15000].reshape(1, 1, 15000)

num_trials = y_small.shape[0]
num_neurons = y_small.shape[1]

dls_test_set = {
    "y": y_small,
    "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "type": torch.zeros(num_trials, dtype=torch.int64)
}

torch.save(dls_test_set, "data/dls_sessions.pt")

print(f"New Shape: {y_small.shape}")
print(f"Points per window: {y_small.shape[2] // 50}")

In [ ]:
!python dunl-compneuro/dunl/train_fiber_loop_acrossneurons.py 

# DUNL 4

In [ ]:
#taking the first 15,000 time points so when we run we do not run out of memory
original_data = torch.as_tensor(dms_matrix, dtype=torch.float32)

if original_data.dim() == 2:
    y_small = original_data[:1, :15000].unsqueeze(0) 
else:
    y_small = original_data.flatten()[:15000].reshape(1, 1, 15000)

num_trials = y_small.shape[0]
num_neurons = y_small.shape[1]

dls_test_set = {
    "y": y_small,
    "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "type": torch.zeros(num_trials, dtype=torch.int64)
}

torch.save(dms_test_set, "data/dms_sessions.pt")

print(f"New Shape: {y_small.shape}")
print(f"Points per window: {y_small.shape[2] // 50}")

In [ ]:
!python dunl-compneuro/dunl/train_fiber_loop_acrossneurons.py 

# DUNL 5

In [ ]:
#initalize lists
dms_sessions = []
dls_sessions = []

#loop through files and load signals
for f in files_100:
    file_path = os.path.join(dir_100, f)
    dms, dls, timestamps = load_fp_signal(file_path)
    dms_sessions.append(dms)
    dls_sessions.append(dls)

#convert lists to object arrays
dms_sessions = np.array(dms_sessions, dtype=object)
dls_sessions = np.array(dls_sessions, dtype=object)

#find the shortest session length to ensure uniform matrix dimensions
min_len = min(len(s) for s in dms_sessions)

#trim sessions to the minimum length and add the 'neuron' dimension (axis 1)
#shape: (num_trials, 1, trial_length)
dms_matrix = np.array([s[:min_len] for s in dms_sessions])[:, np.newaxis, :]
dls_matrix = np.array([s[:min_len] for s in dls_sessions])[:, np.newaxis, :]

#define dimensions for metadata
num_trials, num_neurons, trial_length = dls_matrix.shape

#fefine a helper to package and save as PyTorch tensors immediately
def save_as_tensor_dict(matrix, filename):
    tensor_dict = {
        "y": torch.as_tensor(matrix, dtype=torch.float32),
        "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
        "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
        "type": torch.zeros(num_trials, dtype=torch.int64)
    }
    torch.save(tensor_dict, filename)
    return tensor_dict

#save processed data
dms_sessions_fixed = save_as_tensor_dict(dms_matrix, "data/dms_sessions.pt")
dls_sessions_fixed = save_as_tensor_dict(dls_matrix, "data/dls_sessions.pt")

#verify types and shapes
for key, value in dls_sessions_fixed.items():
    print(f"DLS {key}: {type(value)} | Shape: {value.shape}")

In [ ]:
#taking the first 15,000 time points so when we run we do not run out of memory
original_data = torch.as_tensor(dls_matrix, dtype=torch.float32)

if original_data.dim() == 2:
    y_small = original_data[:1, :15000].unsqueeze(0) 
else:
    y_small = original_data.flatten()[:15000].reshape(1, 1, 15000)

num_trials = y_small.shape[0]
num_neurons = y_small.shape[1]

dls_test_set = {
    "y": y_small,
    "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "type": torch.zeros(num_trials, dtype=torch.int64)
}

torch.save(dls_test_set, "data/dls_sessions.pt")

print(f"New Shape: {y_small.shape}")
print(f"Points per window: {y_small.shape[2] // 50}")

In [ ]:
!python dunl-compneuro/dunl/train_fiber_loop_acrossneurons.py 

# DUNL 6

In [ ]:
#taking the first 15,000 time points so when we run we do not run out of memory
original_data = torch.as_tensor(dms_matrix, dtype=torch.float32)

if original_data.dim() == 2:
    y_small = original_data[:1, :15000].unsqueeze(0) 
else:
    y_small = original_data.flatten()[:15000].reshape(1, 1, 15000)

num_trials = y_small.shape[0]
num_neurons = y_small.shape[1]

dls_test_set = {
    "y": y_small,
    "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "type": torch.zeros(num_trials, dtype=torch.int64)
}

torch.save(dms_test_set, "data/dms_sessions.pt")

print(f"New Shape: {y_small.shape}")
print(f"Points per window: {y_small.shape[2] // 50}")

In [ ]:
!python dunl-compneuro/dunl/train_fiber_loop_acrossneurons.py 